# 服薬仕分けロボット（Genesis / Colab）

AI（VLM）の指示理解と自己確認・やり直しで、薬を曜日×朝夜のケースへ仕分けます。

このノートは **GitHub のリポジトリを取得して実行するだけ** の入口です（コード本体はリポジトリの `src/`）。
**上から順に、同じセッションで実行してください。**

In [ ]:
REPO = 'https://github.com/ma3ma2/pill-sorter.git'
%cd /content
!if [ -d pill-sorter/.git ]; then git -C pill-sorter pull --ff-only; else rm -rf pill-sorter && git clone $REPO pill-sorter; fi
%cd /content/pill-sorter
!ls

In [ ]:
!pip -q install -r /content/pill-sorter/requirements.txt || echo '!! インストールに失敗しました !!'
print('依存の導入おわり')

In [ ]:
!apt-get -qq install -y libosmesa6 libegl1 libgl1 libglu1-mesa fonts-noto-cjk >/dev/null 2>&1
print('描画ライブラリと日本語フォントを準備しました')

In [ ]:
try:
    import genesis as gs
    print('genesis OK:', gs.__version__)
except Exception as e:
    print('genesis が入っていません → 上のインストールセルを確認:', e)

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print('Geminiキーを読み込みました')
except Exception as e:
    print('Geminiキーなし（ルール/画像処理で動作します）:', e)

In [ ]:
import os, sys, torch
sys.path.insert(0, 'src')
from textutil import find_font
from config import CONFIG
print('日本語フォント:', find_font())
print('設定: モデル', CONFIG['gemini']['models'][:2], '/ fps', CONFIG['video']['fps'])
FLAG = '--gpu' if torch.cuda.is_available() else ''
print('GPU:', torch.cuda.is_available())

## 1. 単体テスト（速い・Genesis不要）

In [ ]:
!cd /content/pill-sorter && PYTHONPATH=src python tests/test_units.py

## 2. 動作確認（仕分け）

In [ ]:
!PYTHONPATH=src PYOPENGL_PLATFORM=osmesa python src/sorter.py $FLAG

In [ ]:
from IPython.display import Video, display
display(Video('out/sorter.mp4', embed=True, width=960))

## 3. 確認とやり直し
先に「確認なし」を実行して記録し、次に「確認あり」を実行して動画を作ります（結果カードが実測値になります）。

In [ ]:
!PYTHONPATH=src PYOPENGL_PLATFORM=osmesa python src/sorter_check.py --no-video $FLAG

In [ ]:
!PYTHONPATH=src PYOPENGL_PLATFORM=osmesa python src/sorter_check.py --check --vlm $FLAG

In [ ]:
display(Video('out/check.mp4', embed=True, width=960))

## 4. 提出用動画（約2分）を作成

In [ ]:
!PYTHONPATH=src PYOPENGL_PLATFORM=osmesa python src/make_video.py --seg title $FLAG
!PYTHONPATH=src PYOPENGL_PLATFORM=osmesa python src/make_video.py --seg sort $FLAG
!PYTHONPATH=src PYOPENGL_PLATFORM=osmesa python src/make_video.py --seg check --vlm $FLAG
!PYTHONPATH=src python src/make_video.py --seg result
!PYTHONPATH=src python src/make_video.py --seg concat

In [ ]:
display(Video('out/final.mp4', embed=True, width=960))

## 5. 追加実験（任意・時間がかかる）
確認なし／ありをN回ずつ実行して成功率を集計します（out/summary.json）。

In [ ]:
!PYTHONPATH=src PYOPENGL_PLATFORM=osmesa python tools/run_experiments.py --trials 3 $FLAG

`out/final.mp4` が提出動画、`out/metrics.json`・`out/summary.json` が実験結果、`out/figure_before.png`・`out/figure_after.png` が確認前後の状態図です。